# Korea Relative **Individual** Valuation v1.1
### DuPont 2yr → PBR / PSR / PER 이론 멀티플 → 적정주가 & 괴리율

개별 종목 리스트를 입력하면 종목별 상대가치 평가 후 **6-sheet Excel** 을 저장하는 노트북.
(배치 실행은 `korea_relative_batch_valuation_v1` 사용)

- **모든 입력 변수는 Cell 2 한 곳**에 집중 — Cell 2 만 수정하고 나머지는 위→아래 순서대로 실행
- **★ v1.1 핵심 수정** — 성장률 `g` 를 `calc_sustainable_g()` **단일 함수**로 통합
  - v1 은 `estimate_dupont()`(Re 무시)과 `compute_multiples()`(Re 반영)가 g 를 **서로 다르게**
    계산 → DB 의 `g_est` 와 멀티플에 실제 쓰인 g 가 불일치 → 같은 종목 적정주가 2개 발생
  - v1.1 은 두 곳 모두 같은 함수를 호출하고, `compute_multiples()` 안에서
    `g_est == g_mid` 를 **assert 로 자가검증**
- 클래스 셀 안에 있던 단일 종목 테스트 실행 제거 → 평가는 Cell 9 한 곳에서만 수행

| g 상한 (ROE 구간별) | 값 |
|---|---|
| 저성장 ROE < 15% | `GDP_GROWTH` (4.0%) |
| 중성장 15% ≤ ROE < 30% | `min(8%, Re×0.75)` |
| 고성장 ROE ≥ 30% | `Re × 0.75` |
| 공통 | `Re − g ≥ REL_MIN_SPREAD` 보장 |

## Cell 1 · 경로 자동 감지 (노트북/데스크탑 공용)

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
#  프로젝트 루트(= DATA 폴더의 부모)를 sys.path 에 등록.
#  → 노트북/데스크탑 어느 PC 에서 실행해도 from DATA.* import 가 동일하게 동작
#  ▸ 노트북   : C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\DATA
#  ▸ 데스크탑 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
# ─────────────────────────────────────────────────────────────
_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",
]

def _setup_path() -> str:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root 자동 감지 : {root}")
            return root
    for cand in _CANDIDATE_ROOTS:
        if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "DATA")):
            if cand not in sys.path:
                sys.path.insert(0, cand)
            print(f"[PATH] root 후보 경로 : {cand}")
            return cand
    raise EnvironmentError(
        "DATA 폴더를 찾을 수 없습니다. _CANDIDATE_ROOTS 를 환경에 맞게 수정하세요."
    )

_ROOT = _setup_path()
print(f"[확인] 프로젝트 루트 : {_ROOT}")
print(f"[확인] DATA 경로    : {os.path.join(_ROOT, 'DATA')}")


## Cell 2 · ★ 입력 변수 (여기만 수정하세요)

- `EXPORT_TICKERS` : 평가할 종목 리스트
- `EXPORT_DIR` / `EXPORT_EXCEL` : Excel 저장 폴더·여부
- `SAVE_TO_DB` : DB 동시 저장 여부 (기본 False = Excel/화면 출력만)
- 이하 모델 파라미터 · ROE 구간별 g 상한 · v1.1 가드 상수

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  ★★★ 입력 변수 셀 — 이 노트북의 모든 사용자 입력은 여기 한 곳 ★★★
#  (아래 셀들은 수정할 필요가 없습니다 — 위→아래 순서대로 실행만)
# ═══════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────
#  ① 평가 대상 종목 & 출력
# ─────────────────────────────────────────────────────────────
EXPORT_TICKERS = [          # ★ 평가할 종목 리스트 (A+6자리, 여러 개 가능)
    "A204620",              #   (예시 — 원하는 종목으로 교체)
    # "A005930",            #   삼성전자
    # "A000660",            #   SK하이닉스
]

EXPORT_DIR = (Path(r"C:/reports") if os.name == "nt"
              else Path.home() / "reports")   # Excel 저장 폴더 (없으면 자동 생성)

EXPORT_EXCEL = True         # True → 종목별 6-sheet Excel 저장
SAVE_TO_DB   = False        # True → Excel 출력과 함께 DB(TABLE_RESULT)에도 저장
VERBOSE      = True         # True → 종목별 상세 계산 로그 출력
SHOW_QUALITY = True         # True → 종목별 데이터 품질 리포트 화면 출력

# ─────────────────────────────────────────────────────────────
#  ② DB 테이블 (변경 시에만 수정)
# ─────────────────────────────────────────────────────────────
TABLE_FS        = "korea_fs_data_from_DG"             # 재무제표 원본 (DataGuide)
TABLE_FORECAST  = "korea_revenue_forecast_result"     # 매출 예측 결과
TABLE_MARKETCAP = "ks_listed_company_daily_marketcap" # 일별 시가총액
TABLE_PRICE     = "KSE_Price"                         # 일별 주가
TABLE_RESULT    = "korea_relative_valuation"          # SAVE_TO_DB=True 일 때 저장 테이블
TABLE_QUALITY   = "korea_valuation_quality_log"       # 데이터 품질 로그

# ─────────────────────────────────────────────────────────────
#  ③ 모델 파라미터
# ─────────────────────────────────────────────────────────────
FORECAST_HORIZON     = 8              # 매출 예측 사용 분기 수 (8분기 = 2년)
MIN_HISTORY          = 12             # 회귀에 필요한 최소 분기 수
MIN_REVENUE_QUARTERS = 24             # 평가 최소 조건: 보유 매출 분기 수 (24 = 6년)
WINSORIZE_LIMITS     = (0.05, 0.95)   # 비율 계수 추정 시 winsorize 범위
OLS_MIN_R2           = 0.20           # NOPAT 마진 회귀 채택 최소 R^2
OLS_MIN_SAMPLES      = 12             # NOPAT 마진 회귀 채택 최소 표본 수
GDP_GROWTH           = 0.04           # 한국 장기 GDP 성장률 (저성장 구간 g 상한)

# ─────────────────────────────────────────────────────────────
#  ④ ROE 구간별 g 상한  (★ calc_sustainable_g 이 유일하게 사용)
#     저성장  ROE <  ROE_MID_THRESHOLD   → GDP_GROWTH
#     중성장  ROE <  ROE_HIGH_THRESHOLD  → min(G_CAP_MID_FIXED, Re x G_CAP_RE_RATIO)
#     고성장  ROE >= ROE_HIGH_THRESHOLD  → Re x G_CAP_RE_RATIO
# ─────────────────────────────────────────────────────────────
ROE_MID_THRESHOLD  = 0.15    # 중성장 진입 ROE
ROE_HIGH_THRESHOLD = 0.30    # 고성장 진입 ROE
G_CAP_MID_FIXED    = 0.08    # 중성장 구간 절대 상한
G_CAP_RE_RATIO     = 0.75    # Re 대비 g 상한 비율

# ─────────────────────────────────────────────────────────────
#  ⑤ ★ v1.1 Validity 가드 (FCFF v8 가드와 동일 취지)
# ─────────────────────────────────────────────────────────────
REL_MIN_SPREAD       = 0.005   # Re - g 최소 스프레드.
                               #   0.005 = 구 v1 동작 재현 (기본값)
                               #   0.02  = FCFF v8 의 V8_MIN_TV_SPREAD 와 정렬 (권장)
REL_SANITY_TP_RATIO  = 30.0    # |적정주가/현재가| 허용 배율.
                               #   초과 시 tp_avg 를 제외 (FCFF v8 Sanity Guard 대응)

# ─────────────────────────────────────────────────────────────
#  ⑥ Re (자기자본비용) / ERP
# ─────────────────────────────────────────────────────────────
ERP_METHOD        = "damodaran_floor"  # 'damodaran_floor' | 'kospi_geo_10y'
DAMODARAN_ERP_KR  = 0.07               # Damodaran 한국 ERP
GEO_FLOOR         = 0.07               # KOSPI geo 사용 시 E(Rm) floor
RF_FALLBACK       = 0.035              # BOK API 실패 시 fallback 무위험이자율
RE_FLOOR          = 0.06               # Re 하한 (한국 저금리 고려)
RE_CAP            = 0.18               # Re 상한
BETA_DELTA        = 0.20               # Re 3종 격자 폭 (low/mid/high)

# ─────────────────────────────────────────────────────────────
#  ⑦ Beta 앙상블 & 섹터
# ─────────────────────────────────────────────────────────────
W_INDIVIDUAL = 0.60   # 자체계산 베타(Blume 조정) 가중치
W_SECTOR     = 0.40   # 업종평균 베타 가중치

# 제외 업종 — 회계 특성이 일반 기업과 달라 상대가치 부적절
EXCLUDE_SECTORS = {
    "금융업", "은행", "증권", "보험",
    "부동산", "리츠", "REITs",
    "전기가스업",   # 공익사업
    "운수창고업",   # 해운/항공 사이클성
}

# 섹터별 평균 베타 (한국 시장 경험치)
SECTOR_BETA = {
    "전기전자":     1.25, "IT":         1.30, "서비스업":   1.10,
    "화학":         1.10, "의약품":     0.85, "음식료업":   0.65,
    "유통업":       0.95, "운수장비":   1.15, "기계":       1.20,
    "철강금속":     1.25, "건설업":     1.30, "종이목재":   1.00,
    "섬유의복":     0.95, "비금속광물": 1.00, "의료정밀":   0.90,
    "통신업":       0.80, "제조업":     1.00,
    "Unknown":      1.00,
}

# ─────────────────────────────────────────────────────────────
#  ⑧ 단위 환산 (수정 불필요)
# ─────────────────────────────────────────────────────────────
FS_UNIT_MULTIPLIER        = 1_000        # 재무제표: 천원 → 원
MARKETCAP_UNIT_MULTIPLIER = 1_000_000    # 시가총액: 백만원 → 원

DEBT_KEYS = ["short_term_debt", "current_lt_debt", "bonds",
             "long_term_debt", "lease_liab"]
CASH_KEYS = ["cash", "short_term_invest"]


EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("[OK] 입력 변수 설정 완료")
print(f"  평가 대상 종목   : {EXPORT_TICKERS}")
print(f"  Excel 저장       : {EXPORT_EXCEL}  (폴더: {EXPORT_DIR})")
print(f"  DB 저장          : {SAVE_TO_DB}  (테이블: {TABLE_RESULT})")
print(f"  Re 범위          : [{RE_FLOOR:.0%}, {RE_CAP:.0%}]   GDP={GDP_GROWTH:.1%}")
print(f"  v1.1 가드        : Re-g 최소스프레드 {REL_MIN_SPREAD:.2%}  |  "
      f"Sanity TP/CP {REL_SANITY_TP_RATIO:.0f}배")


## Cell 3 · Import & DB 연결

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Import & DB 연결 — 사용자 입력 없음 (실행만 하면 됩니다)
# ═══════════════════════════════════════════════════════════════

# ── 표준 라이브러리 ──────────────────────────────────────────────
import gc, math, time, traceback, warnings
from datetime import datetime, date, timedelta
from typing import Optional, Dict, Any, List, Tuple

# ── 외부 라이브러리 ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import pymysql
from scipy import stats
from IPython.display import display
import matplotlib
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sqlalchemy import text

# ── 내부 모듈 (Cell 1 경로 자동 감지 덕분에 두 PC 모두 동일하게 import) ──
from DATA.config import get_db_info, get_engine
from DATA.KEYS import KEYS
from DATA.korea_valuation_helpers import (
    setup_project_path, to_dg_ticker, to_price_ticker, get_pymysql_conn,
    DG_ITEM_CODES,
    load_korea_financials_wide, load_korea_revenue_forecast,
    load_korea_marketcap_latest, load_korea_price_series, load_current_price,
    load_kospi_series, get_risk_free_rate, compute_beta_10y,
    estimate_market_return, get_universe_with_min_history,
    DataQualityReport, save_quality_report_to_db, get_evaluation_history,
)
from DATA.universal_ts_forecast_function_v2 import clear_memory


def log(tag: str, msg: str):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}][{tag}] {msg}", flush=True)


# ── DB 연결 ──────────────────────────────────────────────────────
db_info = get_db_info()
engine  = get_engine(db_info)

try:
    with engine.connect() as c:
        c.execute(text("SELECT 1"))
    log("DB", f"연결 성공 host={db_info.get('host')} port={db_info.get('port')}")
except Exception as e:
    log("DB", f"연결 실패: {e}")

print("[OK] Import 완료")
print(f"[설정] FORECAST_HORIZON={FORECAST_HORIZON}Q  ERP_METHOD={ERP_METHOD}")
print(f"[설정] Re 범위 [{RE_FLOOR:.0%}, {RE_CAP:.0%}]  "
      f"g_cap: 저성장 GDP({GDP_GROWTH:.1%}) / 중성장 min({G_CAP_MID_FIXED:.0%}, Re x {G_CAP_RE_RATIO}) "
      f"/ 고성장 Re x {G_CAP_RE_RATIO}")


## Cell 4 · 결과 테이블 초기화 (`SAVE_TO_DB=True` 일 때만)

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  결과 테이블 초기화 (SAVE_TO_DB=True 일 때만) — 사용자 입력 없음
# ═══════════════════════════════════════════════════════════════

if not SAVE_TO_DB:
    log("DB", "SAVE_TO_DB=False → 결과 테이블 초기화 생략 (Excel/화면 출력만 수행)")
else:
    CREATE_SQL = f"""
    CREATE TABLE IF NOT EXISTS `{TABLE_RESULT}` (
      `id`            BIGINT      NOT NULL AUTO_INCREMENT,
      `date`          DATE        NOT NULL COMMENT '평가 실행일',
      `ticker`        VARCHAR(20) NOT NULL,
      `sector`        VARCHAR(50),
      `is_excluded`   TINYINT     DEFAULT 0,
      `sales_y1`      DOUBLE,  `sales_y2`      DOUBLE,
      `npm_y1`        DOUBLE,  `npm_y2`        DOUBLE,
      `at_y1`         DOUBLE,  `at_y2`         DOUBLE,
      `fl_y1`         DOUBLE,  `fl_y2`         DOUBLE,
      `roe_y1`        DOUBLE,  `roe_y2`        DOUBLE,
      `ni_y2`         DOUBLE,  `bve_y2`        DOUBLE,
      `eps_y2`        DOUBLE,  `bps_y2`        DOUBLE,  `sps_y2` DOUBLE,
      `payout_ratio`  DOUBLE,  `g_est`         DOUBLE,
      `beta_raw`      DOUBLE,  `beta_blume`    DOUBLE,
      `beta_sector`   DOUBLE,  `beta_ensemble` DOUBLE,
      `re_low`        DOUBLE,  `re_mid`        DOUBLE,  `re_high` DOUBLE,
      `pbr_theory`    DOUBLE,  `psr_theory`    DOUBLE,  `per_theory` DOUBLE,
      `tp_pbr`        DOUBLE,  `tp_psr`        DOUBLE,  `tp_per`  DOUBLE,
      `tp_avg`        DOUBLE,
      `tp_pbr_low`    DOUBLE,  `tp_pbr_high`   DOUBLE,
      `tp_psr_low`    DOUBLE,  `tp_psr_high`   DOUBLE,
      `current_price` DOUBLE,
      `upside_pbr`    DOUBLE,  `upside_psr`    DOUBLE,
      `upside_per`    DOUBLE,  `upside_avg`    DOUBLE,
      `actual_pbr`    DOUBLE,  `actual_psr`    DOUBLE,  `actual_per` DOUBLE,
      `tax_rate`      DOUBLE,  `npm_r2`        DOUBLE,
      `sanity_ratio`  DOUBLE      COMMENT 'v1.1 TP/CP 배율 (Sanity 가드)',
      `spread_capped` TINYINT     COMMENT 'v1.1 Re-g 최소스프레드 가드 적용 여부',
      `forecast_model` VARCHAR(20) COMMENT '매출 예측 모델명',
      `forecast_date`  DATE        COMMENT '매출 예측 실행일',
      `created_at`    DATETIME DEFAULT CURRENT_TIMESTAMP,
      PRIMARY KEY (`id`),
      UNIQUE KEY uq_main (`ticker`, `date`),
      INDEX idx_ticker (`ticker`),
      INDEX idx_date   (`date`),
      INDEX idx_sector (`sector`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
    """

    conn = get_pymysql_conn(db_info)
    try:
        with conn.cursor() as cur:
            cur.execute(CREATE_SQL)
        conn.commit()
        log("DB", f"테이블 준비 완료: {TABLE_RESULT}")
    finally:
        conn.close()

    # ★ v1.1 신규 컬럼 보강 (이미 있으면 무시)
    _ALTER_COLS = [
        ("sanity_ratio",   "DOUBLE"),
        ("spread_capped",  "TINYINT"),
        ("forecast_model", "VARCHAR(20)"),
        ("forecast_date",  "DATE"),
    ]
    conn = get_pymysql_conn(db_info)
    try:
        with conn.cursor() as cur:
            for _c, _t in _ALTER_COLS:
                try:
                    cur.execute(f"ALTER TABLE `{TABLE_RESULT}` ADD COLUMN `{_c}` {_t}")
                except Exception:
                    pass  # 이미 존재
        conn.commit()
    finally:
        conn.close()


## Cell 5 · 시장 파라미터 (Rf, KOSPI, E(Rm)) — 1회 계산 후 캐싱

모든 종목이 공유하는 시장 공통 파라미터:
- `RF` : BOK API → 10Y 국고채 (실시간)
- `KOSPI_PX` : KOSPI 종합지수 (베타·E(Rm) 계산용)
- `E_RM`, `ERP` : Damodaran 또는 KOSPI geo 10y

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  시장 공통 파라미터 — 사용자 입력 없음
# ═══════════════════════════════════════════════════════════════

# 1. Risk-free rate (BOK API)
RF, RF_SOURCE = get_risk_free_rate(KEYS["BOK"], fallback_rate=RF_FALLBACK)
log("MKT", f"Rf = {RF:.4%}  (source: {RF_SOURCE})")

# 2. KOSPI 시계열 (12년치 → 베타·E(Rm) 모두 사용)
KOSPI_PX = load_kospi_series(
    start_date=(datetime.today() - timedelta(days=365 * 12)).strftime("%Y-%m-%d"))
log("MKT", f"KOSPI {len(KOSPI_PX):,}거래일  "
           f"({KOSPI_PX.index.min().date()} ~ {KOSPI_PX.index.max().date()})")
log("MKT", f"  현재 KOSPI = {KOSPI_PX.iloc[-1]:,.2f}")

# 3. E(Rm) 추정
mkt = estimate_market_return(
    method=ERP_METHOD, rf=RF, kospi_series=KOSPI_PX,
    years=10, damodaran_erp_kr=DAMODARAN_ERP_KR, geo_floor=GEO_FLOOR)
E_RM = mkt["e_rm"]
ERP  = mkt["erp"]
log("MKT", f"E(Rm) = {E_RM:.4%}  ERP = {ERP:.4%}  ({mkt['note']})")

print()
print("=" * 60)
print(f"  Rf    = {RF:>7.3%}")
print(f"  ERP   = {ERP:>7.3%}")
print(f"  E(Rm) = {E_RM:>7.3%}")
print("=" * 60)


## Cell 6 · 섹터 매핑 & `load_korea_revenue_forecast` 패치

- 섹터: FDR `StockListing('KRX')` 1회 로드 후 dict 캐싱
- 패치: 구스키마 `created_at` 버그(매출 forecast 상수 복제) 수정판.
  `updated_at` 버전 선택 + 3중 검증(분기 수·연속성·상수 시계열)

> ⚠️ Cell 3(import 셀)을 다시 실행하면 패치가 풀리므로, 이 셀도 반드시 재실행하세요.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  섹터 매핑 (FDR StockListing) — 사용자 입력 없음
#  FDR 결과에 섹터 컬럼이 없으면 전 종목 'Unknown' → 섹터 베타 앙상블만 영향
# ═══════════════════════════════════════════════════════════════
SECTOR_LOOKUP: Dict[str, Dict[str, str]] = {}
try:
    import FinanceDataReader as fdr
    listing = fdr.StockListing("KRX")
    code_col   = next((c for c in ["Code", "Symbol", "code", "ticker"]
                       if c in listing.columns), None)
    sector_col = next((c for c in ["Sector", "Industry", "업종", "sector", "industry"]
                       if c in listing.columns), None)
    indu_col   = next((c for c in ["Industry", "IndustryCode", "업종명"]
                       if c in listing.columns and c != sector_col), None)
    name_col   = next((c for c in ["Name", "name", "기업명"]
                       if c in listing.columns), None)
    if code_col and sector_col:
        for _, row in listing.iterrows():
            code = str(row[code_col]).strip().zfill(6)
            SECTOR_LOOKUP[code] = {
                "sector":   str(row[sector_col]) if pd.notna(row[sector_col]) else "Unknown",
                "industry": str(row[indu_col]) if indu_col and pd.notna(row.get(indu_col)) else "",
                "name":     str(row[name_col]) if name_col and pd.notna(row.get(name_col)) else "",
            }
        log("SECTOR", f"FDR StockListing 로드 → {len(SECTOR_LOOKUP):,}개 종목 섹터 매핑")
    else:
        log("SECTOR", f"[WARN] FDR 컬럼명 불일치 (col={list(listing.columns)}) → 전체 Unknown")
except Exception as e:
    log("SECTOR", f"[WARN] FDR StockListing 실패: {e} → 전체 Unknown")


def get_sector(ticker) -> Dict[str, str]:
    """ticker → {'sector', 'industry', 'name'} (기본 Unknown)."""
    code = to_price_ticker(ticker)
    return SECTOR_LOOKUP.get(code, {"sector": "Unknown", "industry": "", "name": ""})


print(f"[OK] 섹터 매핑 {len(SECTOR_LOOKUP):,}개")


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  PATCH 셀 · load_korea_revenue_forecast 교체
#  (구스키마 created_at 버그 → 매출 forecast 상수 복제 문제 수정)
#
#  ▸ 붙여넣을 위치 : korea_fcff_dcf_valuation_v7.ipynb 의
#                    import 셀(DATA.korea_valuation_helpers import) "바로 다음".
#                    KoreaDCFModel 실행(Cell 9/10) 전이면 어디든 OK.
#  ▸ 원리          : 노트북 전역의 load_korea_revenue_forecast 이름을
#                    아래 수정판으로 재바인딩 → KoreaDCFModel.load_sales 가
#                    호출 시점에 이 패치판을 사용.
#  ▸ 주의          : import 셀을 다시 실행하면 패치가 풀리므로,
#                    import 셀 재실행 후에는 이 셀도 재실행할 것.
#
#  ▸ 수정 내용 (korea_revenue_analysis_notebook_v3 의 fetch_forecast 와 동일 원리)
#    1) 버전 식별: MAX(created_at) → MAX(updated_at) (자동 감지, 없으면 created_at)
#       - 구스키마 UNIQUE KEY(date,ticker,indicator) 에서는 재예측 시
#         겹치는 분기의 created_at 이 갱신되지 않아 '새로 생긴 분기 1개'만
#         최신 버전으로 잡히는 문제가 있었음 (예: A031330 → 2028Q1 한 분기).
#    2) 같은 분기 중복 시 가장 최근 갱신분만 사용.
#    3) 검증 추가 — 아래 셋 중 하나라도 걸리면 ValueError 로 즉시 차단
#       (잘못된 valuation 이 DB 에 저장되는 것을 막기 위함):
#       a. 분기 수 < horizon
#       b. 분기 불연속 (중간 분기 누락)
#       c. 전 분기 동일값 (상수 시계열 — 이번 버그의 증상)
# ═══════════════════════════════════════════════════════════════
from sqlalchemy import text as _sa_text

_FC_MODEL_PRIORITY = ("Ensemble", "SARIMA", "ETS", "Theta")
_FC_RECENCY_CACHE: dict = {}


def _fc_recency_col(table_name: str) -> str:
    """updated_at 컬럼이 있으면 그것을, 없으면 created_at 을 버전 기준으로 사용."""
    if table_name in _FC_RECENCY_CACHE:
        return _FC_RECENCY_CACHE[table_name]
    sql = ("SELECT COLUMN_NAME FROM information_schema.COLUMNS "
           "WHERE TABLE_SCHEMA = :db AND TABLE_NAME = :tbl")
    with engine.connect() as conn:
        cols = {r[0].lower() for r in conn.execute(
            _sa_text(sql), {"db": db_info["database"], "tbl": table_name}
        ).fetchall()}
    rc = "updated_at" if "updated_at" in cols else "created_at"
    _FC_RECENCY_CACHE[table_name] = rc
    if rc == "created_at":
        log("PATCH", f"[주의] {table_name} 에 updated_at 없음 → created_at 기준. "
                     "구스키마면 일부 종목 분기 누락 가능 — updated_at 추가 권장.")
    return rc


def load_korea_revenue_forecast(ticker: str,
                                db_info: dict,
                                table_name: str = "korea_revenue_forecast_result",
                                horizon: int = 8):
    """[PATCHED v7-fix] 최신 '실행 버전' 전체 분기를 안전하게 로드.

    Returns
    -------
    (pd.Series, str, str)
        forecast : 천원 단위, index=분기 date (호출측에서 ×1000 변환)
        model    : 사용된 indicator (Ensemble 우선)
        run_date : 예측 실행 버전 날짜 (YYYY-MM-DD)
    """
    rc = _fc_recency_col(table_name)

    df, used_model, run_date = None, "", None
    for model in _FC_MODEL_PRIORITY:
        # indicator 가 'Ensemble' 형태든 '매출액(천원)_Ensemble' 형태든 모두 매칭
        with engine.connect() as conn:
            row = conn.execute(_sa_text(
                f"SELECT MAX(DATE({rc})) FROM {table_name} "
                f"WHERE ticker = :t AND (indicator = :m OR indicator LIKE :ml)"
            ), {"t": ticker, "m": model, "ml": f"%\\_{model}"}).fetchone()
        if row is None or row[0] is None:
            continue
        run_date = str(row[0])
        cand = pd.read_sql(_sa_text(
            f"SELECT date, value, {rc} AS run_ts FROM {table_name} "
            f"WHERE ticker = :t AND (indicator = :m OR indicator LIKE :ml) "
            f"  AND DATE({rc}) = :fd "
            f"ORDER BY date"
        ), engine, params={"t": ticker, "m": model,
                           "ml": f"%\\_{model}", "fd": run_date})
        if not cand.empty:
            df, used_model = cand, model
            break

    if df is None:
        return pd.Series(dtype=float), "", None

    df["date"]   = pd.to_datetime(df["date"])
    df["run_ts"] = pd.to_datetime(df["run_ts"])
    df = (df.sort_values(["date", "run_ts"])
            .drop_duplicates(subset=["date"], keep="last"))
    fc = df.set_index("date")["value"].astype(float).sort_index()

    # ── 검증 a: 분기 수 ───────────────────────────────────────
    if len(fc) < horizon:
        raise ValueError(
            f"[{ticker}] forecast 분기 {len(fc)}개 < horizon {horizon} "
            f"(version={rc} {run_date}, indicator={used_model}). "
            f"구스키마 created_at 잔존 가능성 — 해당 종목 재예측 또는 "
            f"테이블 updated_at 마이그레이션 필요."
        )
    fc = fc.iloc[:horizon]

    # ── 검증 b: 분기 연속성 ──────────────────────────────────
    per = fc.index.to_period("Q")
    gaps = (per[1:].astype("int64") - per[:-1].astype("int64"))
    if (gaps != 1).any():
        bad = [str(per[i + 1]) for i, g in enumerate(gaps) if g != 1]
        raise ValueError(
            f"[{ticker}] forecast 분기 불연속: {bad} "
            f"(version={rc} {run_date}). 해당 종목 재예측 필요."
        )

    # ── 검증 c: 상수 시계열 (이번 버그의 증상) ────────────────
    if fc.nunique() == 1:
        raise ValueError(
            f"[{ticker}] forecast {horizon}개 분기가 모두 동일값"
            f"({fc.iloc[0]:,.0f}천원) — 단일 분기 복제 의심. "
            f"(version={rc} {run_date}). 해당 종목 재예측 필요."
        )

    return fc, used_model, run_date


# log("PATCH", "load_korea_revenue_forecast → updated_at 버전 선택 + 3중 검증 패치 적용 완료")


## Cell 7 · `KoreaRelValModel` v1.1 클래스

★ `calc_sustainable_g()` 단일화 · Re−g 스프레드 가드 · Sanity 가드(TP/CP) 포함.
클래스 정의만 수행하며 **평가는 실행하지 않습니다**.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  KoreaRelValModel v1.1 — DuPont 2yr → PBR/PSR/PER 상대가치
#  ─────────────────────────────────────────────────────────────
#  v1 → v1.1 변경 (★ 평가 이중화 제거 / 단일화)
#    ① calc_sustainable_g() 를 모듈 레벨 단일 함수로 분리
#       - 기존 v1 은 estimate_dupont() 와 compute_multiples() 가
#         g 를 서로 다른 공식으로 계산 → DB 의 g_est 와 실제
#         멀티플에 쓰인 g 가 불일치 (같은 종목 두 값 발생).
#       - v1.1 은 두 곳 모두 이 함수 하나만 호출 → 항상 동일.
#    ② Re−g 최소 스프레드 가드 (REL_MIN_SPREAD) 를 g 상한에 반영
#       - 분자(ROE−g)/분모(Re−g) 가 같은 g 를 쓰도록 정합성 확보
#    ③ Sanity 가드 : |TP/CP| > REL_SANITY_TP_RATIO 이면 평가 제외
#    ④ 클래스 셀 내부의 단일 종목 테스트 실행 제거
#       (개별 평가는 individual 노트북, 배치는 batch 노트북 담당)
# ═══════════════════════════════════════════════════════════════


def calc_sustainable_g(roe: float, payout: float, re: float) -> Dict[str, Any]:
    """지속가능 성장률 g 산출 — ★ 이 노트북에서 g 계산의 유일한 출처.

    g_raw = ROE × 유보율(1 − payout)
    ROE 구간별 상한:
        저성장 ROE < ROE_MID_THRESHOLD   → GDP_GROWTH
        중성장 ROE < ROE_HIGH_THRESHOLD  → min(G_CAP_MID_FIXED, Re × G_CAP_RE_RATIO)
        고성장 ROE ≥ ROE_HIGH_THRESHOLD  → Re × G_CAP_RE_RATIO
    추가로 Re − g ≥ REL_MIN_SPREAD 를 항상 보장 (Gordon 분모 발산 방지).

    Returns
    -------
    dict : {'g', 'g_raw', 'g_cap', 'spread_capped'}
    """
    if roe is None or (isinstance(roe, float) and np.isnan(roe)):
        return {"g": 0.0, "g_raw": 0.0, "g_cap": 0.0, "spread_capped": False}

    retention = max(1.0 - payout, 0.0)
    g_raw = roe * retention

    if roe >= ROE_HIGH_THRESHOLD:
        g_cap = re * G_CAP_RE_RATIO
    elif roe >= ROE_MID_THRESHOLD:
        g_cap = min(G_CAP_MID_FIXED, re * G_CAP_RE_RATIO)
    else:
        g_cap = GDP_GROWTH

    # ★ Re − g 최소 스프레드 보장
    g_cap_spread = re - REL_MIN_SPREAD
    spread_capped = bool(g_cap > g_cap_spread)
    g_cap = min(g_cap, g_cap_spread)

    g = float(np.clip(g_raw, 0.0, max(g_cap, 0.0)))
    return {"g": g, "g_raw": float(g_raw), "g_cap": float(g_cap),
            "spread_capped": spread_capped}


class KoreaRelValModel:
    """한국 주식 상대가치 3종(PBR/PSR/PER) 적정주가 평가.

    US v4 로직 기반 (Gordon Growth + ROE 구간별 g cap + NOPAT margin OLS).
    데이터 소스는 DataGuide DB.
    """

    def __init__(self, ticker, engine, db_info, rf, e_rm, kospi_series,
                 verbose=False):
        self.ticker_dg    = to_dg_ticker(ticker)
        self.ticker_price = to_price_ticker(ticker)
        self.engine       = engine
        self.db_info      = db_info
        self.rf           = rf
        self.e_rm         = e_rm
        self.erp          = e_rm - rf
        self.kospi        = kospi_series
        self.verbose      = verbose

        self._sales_actual   = None
        self._sales_forecast = None
        self._used_model     = ""
        self._forecast_date  = None
        self._fs_wide        = None
        self._sector         = "Unknown"
        self._is_excluded    = False

        # ★ v1.1: Excel/리포트용 중간 산출물 보관
        self._beta_re   = None
        self._dupont    = None
        self._mults     = None
        self.valuation  = None

        self.report = DataQualityReport(ticker=self.ticker_dg)

    @staticmethod
    def _winsorize(s):
        s = s.dropna()
        if len(s) < 4:
            return s
        lo, hi = s.quantile(WINSORIZE_LIMITS[0]), s.quantile(WINSORIZE_LIMITS[1])
        return s.clip(lo, hi)

    @staticmethod
    def _n(val):
        if val is None:
            return None
        try:
            f = float(val)
            return None if (np.isnan(f) or np.isinf(f)) else f
        except (TypeError, ValueError):
            return val

    # ─────────────────────────────────────────────────────────
    # 1. Sales 로드
    # ─────────────────────────────────────────────────────────
    def load_sales(self):
        wide = load_korea_financials_wide(
            self.ticker_dg, self.db_info, table_name=TABLE_FS,
            item_keys=["revenue"], fillna_zero=False)
        actual = wide["revenue"].dropna() * FS_UNIT_MULTIPLIER
        if actual.empty or len(actual) < MIN_REVENUE_QUARTERS:
            self.report.add("revenue_actual", "missing", n_obs=len(actual),
                            note=f"actual {len(actual)}Q < {MIN_REVENUE_QUARTERS}")
            raise ValueError(f"[{self.ticker_dg}] 매출 actual 부족 ({len(actual)}Q)")
        self.report.add("revenue_actual", "ok", n_obs=len(actual))

        forecast, model_name, ca = load_korea_revenue_forecast(
            self.ticker_dg, self.db_info, table_name=TABLE_FORECAST,
            horizon=FORECAST_HORIZON)
        if forecast.empty:
            self.report.add("revenue_forecast", "missing", n_obs=0,
                            note="korea_revenue_forecast_result 없음")
            raise ValueError(f"[{self.ticker_dg}] 매출 forecast 없음")
        forecast = forecast * FS_UNIT_MULTIPLIER
        self.report.add("revenue_forecast", "ok", n_obs=len(forecast),
                        note=f"model={model_name}")

        self._sales_actual   = actual
        self._sales_forecast = forecast.iloc[:FORECAST_HORIZON]
        self._used_model     = model_name
        self._forecast_date  = ca
        if self.verbose:
            log(self.ticker_dg,
                f"Sales actual={len(actual)}Q forecast={len(self._sales_forecast)}Q "
                f"({model_name})")
        return self

    # ─────────────────────────────────────────────────────────
    # 2. 재무제표 (DataGuide)
    # ─────────────────────────────────────────────────────────
    def load_financials(self):
        wide = load_korea_financials_wide(
            self.ticker_dg, self.db_info, table_name=TABLE_FS,
            item_keys=None, fillna_zero=False)
        if wide.empty:
            raise ValueError(f"[{self.ticker_dg}] FS 데이터 없음")

        non_share = [c for c in wide.columns
                     if c not in ("shares_treasury_adj", "shares_common")]
        wide[non_share] = wide[non_share] * FS_UNIT_MULTIPLIER

        if "operating_income" not in wide.columns or \
           wide["operating_income"].dropna().empty:
            self.report.add("operating_income", "missing", n_obs=0)
            raise ValueError(f"[{self.ticker_dg}] operating_income 없음")
        self.report.add("operating_income", "ok",
                        n_obs=int(wide["operating_income"].notna().sum()))

        self._fs_wide = wide
        if self.verbose:
            log(self.ticker_dg, f"FS wide shape={wide.shape}")
        return self

    # ─────────────────────────────────────────────────────────
    # 3. 섹터 로드
    # ─────────────────────────────────────────────────────────
    def load_sector(self):
        info = get_sector(self.ticker_dg)
        sector = info["sector"]
        self._sector      = sector
        self._is_excluded = sector in EXCLUDE_SECTORS
        self.report.add("sector", "ok" if sector != "Unknown" else "fallback_zero",
                        note=f"sector={sector}  excluded={self._is_excluded}")
        if self.verbose:
            excl = " [EXCLUDED]" if self._is_excluded else ""
            log(self.ticker_dg, f"sector={sector}{excl}")
        return self

    # ─────────────────────────────────────────────────────────
    # 4. Beta & Re (자체 계산)
    # ─────────────────────────────────────────────────────────
    def estimate_beta_re(self):
        """KSE_Price vs KOSPI 10년 회귀 + Blume → 업종평균 앙상블.
        Re 3종(low/mid/high) : β_ensemble ± BETA_DELTA → CAPM."""
        info = compute_beta_10y(
            self.ticker_dg, self.db_info, kospi_series=self.kospi,
            years=10, min_obs=750)

        beta_raw = info["beta_raw"] if not np.isnan(info["beta_raw"]) else \
                   SECTOR_BETA.get(self._sector, 1.0)
        beta_blume  = 0.67 * abs(beta_raw) + 0.33 if not np.isnan(beta_raw) else 1.0
        beta_sector = SECTOR_BETA.get(self._sector, 1.0)
        beta_ens    = W_INDIVIDUAL * beta_blume + W_SECTOR * beta_sector

        if np.isnan(info["beta_raw"]):
            self.report.add("beta", "fallback_median", n_obs=info["n_obs"],
                            value=beta_sector,
                            note=f"자체 베타 실패 → 업종({self._sector}) 평균")
        else:
            self.report.add("beta", "ok", n_obs=info["n_obs"],
                            value=beta_blume, r2=info["r_squared"],
                            note=f"raw={info['beta_raw']:.3f}")

        def _re(b):
            return float(np.clip(self.rf + b * self.erp, RE_FLOOR, RE_CAP))

        result = {
            "beta_raw": float(beta_raw), "beta_blume": float(beta_blume),
            "beta_sector": float(beta_sector), "beta_ensemble": float(beta_ens),
            "re_low":  _re(beta_ens - BETA_DELTA),
            "re_mid":  _re(beta_ens),
            "re_high": _re(beta_ens + BETA_DELTA),
        }
        if self.verbose:
            log(self.ticker_dg,
                f"Beta raw={beta_raw:.3f} blume={beta_blume:.3f} "
                f"sector({self._sector})={beta_sector:.2f} ens={beta_ens:.3f} | "
                f"Re low={result['re_low']:.3%} mid={result['re_mid']:.3%} "
                f"high={result['re_high']:.3%}")
        self._beta_re = result
        return result

    # ─────────────────────────────────────────────────────────
    # 5. DuPont 2yr (NOPAT margin 기반)
    # ─────────────────────────────────────────────────────────
    def estimate_dupont(self, beta_re):
        """★ v1.1: beta_re 를 인자로 받아 g_est 를 calc_sustainable_g() 로 산출.
        (v1 은 Re 를 무시한 별도 공식을 써서 compute_multiples 의 g 와 불일치했음)"""
        wide = self._fs_wide

        # ── 실효세율 ────────────────────────────────────────
        tax = 0.22  # 한국 법정세율 fallback
        if "pretax_income" in wide.columns and "tax_expense" in wide.columns:
            df_tax = wide[["pretax_income", "tax_expense"]].dropna()
            df_tax = df_tax[df_tax["pretax_income"] > 0]
            if not df_tax.empty:
                rates = (df_tax["tax_expense"] / df_tax["pretax_income"]).clip(0, 0.40)
                tax = float(rates.median())
                self.report.add("tax_rate", "ok", n_obs=len(rates), value=tax)
            else:
                self.report.add("tax_rate", "fallback_zero", value=0.22,
                                note="pretax 양수 분기 없음 → 법정세율")
        else:
            self.report.add("tax_rate", "fallback_zero", value=0.22,
                            note="pretax/tax 컬럼 없음")

        # ── NOPAT margin OLS (revenue vs NOPAT = OI×(1-t)) ─
        if "revenue" in wide.columns and "operating_income" in wide.columns:
            df = wide[["revenue", "operating_income"]].dropna()
            df = df[df["revenue"] > 0].copy()
            df["nopat"] = df["operating_income"] * (1.0 - tax)

            nopat_series = self._winsorize((df["nopat"] / df["revenue"]).dropna())
            npm_median = float(nopat_series.median()) if not nopat_series.empty else 0.08
            npm_coef   = npm_median
            npm_r2     = -1.0

            if len(df) >= OLS_MIN_SAMPLES:
                slope, _, r, _, _ = stats.linregress(df["revenue"], df["nopat"])
                if r ** 2 >= OLS_MIN_R2 and slope >= 0:
                    npm_coef = float(slope)
                    npm_r2   = float(r ** 2)
                    self.report.add("nopat_margin", "ok", n_obs=len(df),
                                    value=npm_coef, r2=npm_r2, note="OLS slope")
                else:
                    self.report.add("nopat_margin", "fallback_median",
                                    n_obs=len(df), value=npm_median, r2=float(r ** 2),
                                    note=f"R2={r ** 2:.2f} < {OLS_MIN_R2} → median")
            else:
                self.report.add("nopat_margin", "fallback_median",
                                n_obs=len(df), value=npm_median,
                                note=f"n={len(df)} < {OLS_MIN_SAMPLES}")
        else:
            npm_coef, npm_median, npm_r2 = 0.08, 0.08, -1.0
            self.report.add("nopat_margin", "fallback_zero", value=0.08,
                            note="revenue/OI 컬럼 없음")

        # ── Asset Turnover (TTM 기준) ───────────────────────
        at_median = 0.70
        if "revenue" in wide.columns and "total_assets" in wide.columns:
            df_at = wide[["revenue", "total_assets"]].dropna().sort_index()
            df_at = df_at[df_at["total_assets"] > 0]
            if len(df_at) >= 4:
                df_at["ttm"] = df_at["revenue"].rolling(4).sum()
                atm = df_at.dropna(subset=["ttm"])
                at_ratios = self._winsorize((atm["ttm"] / atm["total_assets"]).dropna())
                if not at_ratios.empty:
                    at_median = float(at_ratios.median())
                    self.report.add("asset_turnover", "ok",
                                    n_obs=len(at_ratios), value=at_median)
            else:
                self.report.add("asset_turnover", "fallback_zero",
                                value=at_median, note="total_assets 4분기 미만")
        else:
            self.report.add("asset_turnover", "fallback_zero", value=at_median,
                            note="total_assets 컬럼 없음")

        # ── Financial Leverage ──────────────────────────────
        fl_median = 2.5
        if "total_assets" in wide.columns and "total_equity" in wide.columns:
            df_fl = wide[["total_assets", "total_equity"]].dropna()
            df_fl = df_fl[(df_fl["total_assets"] > 0) & (df_fl["total_equity"] > 0)]
            if not df_fl.empty:
                fl_ratios = self._winsorize(
                    (df_fl["total_assets"] / df_fl["total_equity"]).dropna())
                fl_median = float(np.clip(fl_ratios.median(), 1.0, 20.0))
                self.report.add("financial_leverage", "ok",
                                n_obs=len(fl_ratios), value=fl_median)
            else:
                self.report.add("financial_leverage", "fallback_zero",
                                value=fl_median, note="자본/자산 양수 분기 없음")
        else:
            self.report.add("financial_leverage", "fallback_zero", value=fl_median)

        # ── Annual Sales 2yr ────────────────────────────────
        fc_q = self._sales_forecast
        annual_sales = []
        for yr in range(0, min(len(fc_q), 8), 4):
            annual_sales.append(float(fc_q.iloc[yr:yr + 4].sum()))
        while len(annual_sales) < 2:
            annual_sales.append(annual_sales[-1] if annual_sales else 0)

        # ── BVE 시작값 ──────────────────────────────────────
        if "total_equity" in wide.columns:
            eq_ser = wide["total_equity"].dropna()
            bve0 = float(eq_ser.iloc[-1]) if not eq_ser.empty else 1e10
        else:
            bve0 = 1e10
        if bve0 <= 0:
            bve0 = 1e10

        # ── Shares (자사주차감 우선) ────────────────────────
        shares = np.nan
        for col in ["shares_treasury_adj", "shares_common"]:
            if col in wide.columns:
                s = wide[col].dropna()
                s = s[s > 0]
                if not s.empty:
                    shares = float(s.iloc[-1])
                    break
        if np.isnan(shares):
            self.report.add("shares", "missing", n_obs=0)
        else:
            self.report.add("shares", "ok", value=shares)

        # ── Payout ratio (배당만 사용) ──────────────────────
        payout_med = 0.15
        if "dividends_paid" in wide.columns:
            div_ser = wide["dividends_paid"].abs().fillna(0)
            nopat_ser = (wide["operating_income"] * (1.0 - tax)).abs()
            valid = (nopat_ser > 0) & (div_ser >= 0)
            if valid.sum() >= 4:
                po = (div_ser[valid] / nopat_ser[valid]).clip(0, 3.0)
                po_recent = po.iloc[-8:] if len(po) >= 8 else po
                payout_med = float(po_recent.median())
                self.report.add("payout_ratio", "ok", n_obs=int(valid.sum()),
                                value=payout_med, note="dividends_paid/NOPAT")
            else:
                self.report.add("payout_ratio", "fallback_zero",
                                value=payout_med, note=f"valid {valid.sum()} < 4")
        else:
            self.report.add("payout_ratio", "fallback_zero", value=payout_med,
                            note="dividends_paid 컬럼 없음")

        # ── 2yr DuPont 루프 ─────────────────────────────────
        results_yr = []
        bve = bve0
        for sales in annual_sales[:2]:
            nopat_est = npm_coef * sales
            npm       = nopat_est / sales if sales > 0 else 0.0
            roe       = float(np.clip(npm * at_median * fl_median, -0.99, 3.0))
            retention = max(1.0 - payout_med, -0.5)
            bve_end   = bve + nopat_est * retention
            results_yr.append({"sales": sales, "ni": nopat_est, "npm": npm,
                               "at": at_median, "fl": fl_median, "roe": roe,
                               "bve": bve_end})
            bve = bve_end

        y1, y2 = results_yr[0], results_yr[1]
        eps_y2 = y2["ni"]    / shares if (not np.isnan(shares) and shares > 0) else np.nan
        bps_y2 = y2["bve"]   / shares if (not np.isnan(shares) and shares > 0) else np.nan
        sps_y2 = y2["sales"] / shares if (not np.isnan(shares) and shares > 0) else np.nan

        # ── ★ v1.1: g 는 calc_sustainable_g() 단일 함수로만 산출 ──
        g_info = calc_sustainable_g(y2["roe"], payout_med, beta_re["re_mid"])
        g_est  = g_info["g"]
        if g_info["spread_capped"]:
            self.report.add("g_est", "fallback_median", value=g_est,
                            note=f"Re-g 최소스프레드({REL_MIN_SPREAD:.1%}) 가드 적용")
        else:
            self.report.add("g_est", "ok", value=g_est,
                            note=f"g_raw={g_info['g_raw']:.3f} cap={g_info['g_cap']:.3f}")

        if self.verbose:
            ols_tag = f"OLS(R2={npm_r2:.2f})" if npm_r2 >= OLS_MIN_R2 else "median"
            eps_s = f"{eps_y2:,.0f}원" if not np.isnan(eps_y2) else "N/A"
            bps_s = f"{bps_y2:,.0f}원" if not np.isnan(bps_y2) else "N/A"
            sps_s = f"{sps_y2:,.0f}원" if not np.isnan(sps_y2) else "N/A"
            log(self.ticker_dg,
                f"DuPont Y2 ROE={y2['roe']:.1%} NOPATm={y2['npm']:.1%}({ols_tag}) "
                f"AT={at_median:.2f} FL={fl_median:.1f} tax={tax:.1%} | "
                f"EPS={eps_s} BPS={bps_s} SPS={sps_s} g={g_est:.2%} "
                f"payout={payout_med:.1%}")

        out = {
            "npm_y1": y1["npm"], "npm_y2": y2["npm"],
            "at_y1":  y1["at"],  "at_y2":  y2["at"],
            "fl_y1":  y1["fl"],  "fl_y2":  y2["fl"],
            "roe_y1": y1["roe"], "roe_y2": y2["roe"],
            "sales_y1": y1["sales"], "sales_y2": y2["sales"],
            "ni_y2":   y2["ni"],   "bve_y2": y2["bve"],
            "eps_y2":  eps_y2,     "bps_y2": bps_y2, "sps_y2": sps_y2,
            "payout_ratio": payout_med, "g_est": g_est,
            "g_raw": g_info["g_raw"], "g_cap": g_info["g_cap"],
            "g_spread_capped": g_info["spread_capped"],
            "shares": shares, "tax_rate": tax, "npm_r2": npm_r2,
            "bve0": bve0,
        }
        self._dupont = out
        return out

    # ─────────────────────────────────────────────────────────
    # 6. 이론 멀티플 & 적정가
    # ─────────────────────────────────────────────────────────
    def compute_multiples(self, dupont, beta_re):
        roe    = dupont["roe_y2"]
        npm    = dupont["npm_y2"]
        payout = dupont["payout_ratio"]
        eps, bps, sps = dupont["eps_y2"], dupont["bps_y2"], dupont["sps_y2"]
        re_mid, re_low, re_high = (beta_re["re_mid"], beta_re["re_low"],
                                   beta_re["re_high"])

        def _mult(re_):
            # ★ v1.1: estimate_dupont 과 동일한 g 함수 사용 (이중 계산 제거)
            g  = calc_sustainable_g(roe, payout, re_)["g"]
            sp = max(re_ - g, REL_MIN_SPREAD)
            pbr = float(np.clip((roe - g) / sp, 0.1, 100.0)) if roe > g else 0.1
            psr = float(np.clip(pbr * max(npm, 0) / roe, 0.01, 50.0)) if roe > 0 else np.nan
            per = float(np.clip(pbr / roe, 1.0, 200.0)) if roe > 0 else np.nan
            return pbr, psr, per, g

        pbr_m, psr_m, per_m, g_mid = _mult(re_mid)
        pbr_l, psr_l, _, _         = _mult(re_low)
        pbr_h, psr_h, _, _         = _mult(re_high)

        # ── 정합성 자가검증: g_mid 는 DB 에 저장되는 g_est 와 반드시 동일 ──
        if abs(g_mid - dupont["g_est"]) > 1e-12:
            raise AssertionError(
                f"[{self.ticker_dg}] g 불일치: g_est={dupont['g_est']:.6f} "
                f"vs g_mid={g_mid:.6f} — calc_sustainable_g 단일화 위반")

        def _tp(mult, base):
            if np.isnan(base) or base <= 0 or np.isnan(mult):
                return np.nan
            return float(mult * base)

        tp_pbr = _tp(pbr_m, bps)
        tp_psr = _tp(psr_m, sps)
        tp_per = _tp(per_m, eps)
        tp_pbr_low, tp_pbr_hi = _tp(pbr_l, bps), _tp(pbr_h, bps)
        tp_psr_low, tp_psr_hi = _tp(psr_l, sps), _tp(psr_h, sps)

        valid = [x for x in [tp_pbr, tp_psr, tp_per] if not np.isnan(x) and x > 0]
        tp_avg = float(np.mean(valid)) if valid else np.nan

        # ── 현재가 ──────────────────────────────────────────
        cp = load_current_price(self.ticker_dg, self.db_info, table_name=TABLE_PRICE)
        if cp is None:
            self.report.add("current_price", "missing", n_obs=0)
            cp = np.nan

        # ── ★ v1.1 Sanity 가드 (FCFF v8 ④ 와 동일 취지) ────
        sanity_ratio = (tp_avg / cp) if (not np.isnan(tp_avg) and
                                         not np.isnan(cp) and cp > 0) else np.nan
        sanity_ok = True
        if not np.isnan(sanity_ratio):
            if sanity_ratio > REL_SANITY_TP_RATIO or sanity_ratio < 1.0 / REL_SANITY_TP_RATIO:
                sanity_ok = False
                self.report.add("sanity_guard", "missing", value=sanity_ratio,
                                note=f"TP/CP={sanity_ratio:.1f}배 "
                                     f"(허용 1/{REL_SANITY_TP_RATIO:.0f}~{REL_SANITY_TP_RATIO:.0f}배) → 평가 제외")
                if self.verbose:
                    log(self.ticker_dg,
                        f"[SANITY FAIL] TP/CP={sanity_ratio:.1f}배 → tp_avg 제외")
                tp_avg = np.nan

        def _up(tp):
            return ((tp / cp - 1) * 100
                    if (not np.isnan(tp) and not np.isnan(cp) and cp > 0) else np.nan)

        actual_pbr = float(cp / bps) if (not np.isnan(cp) and not np.isnan(bps) and bps > 0) else np.nan
        actual_psr = float(cp / sps) if (not np.isnan(cp) and not np.isnan(sps) and sps > 0) else np.nan
        actual_per = float(cp / eps) if (not np.isnan(cp) and not np.isnan(eps) and eps > 0) else np.nan

        if self.verbose:
            tp_str = f"{tp_avg:,.0f}원" if not np.isnan(tp_avg) else "N/A"
            cp_str = f"{cp:,.0f}원" if not np.isnan(cp) else "N/A"
            up_str = f"{_up(tp_avg):+.1f}%" if not np.isnan(_up(tp_avg)) else "N/A"
            log(self.ticker_dg,
                f"g={g_mid:.2%} PBR={pbr_m:.2f}x PSR={psr_m:.2f}x PER={per_m:.1f}x | "
                f"TP_avg={tp_str} CP={cp_str} Up={up_str}")

        out = {
            "pbr_theory": pbr_m, "psr_theory": psr_m, "per_theory": per_m,
            "tp_pbr": tp_pbr, "tp_psr": tp_psr, "tp_per": tp_per, "tp_avg": tp_avg,
            "tp_pbr_low": tp_pbr_low, "tp_pbr_high": tp_pbr_hi,
            "tp_psr_low": tp_psr_low, "tp_psr_high": tp_psr_hi,
            "current_price": cp,
            "upside_pbr": _up(tp_pbr), "upside_psr": _up(tp_psr),
            "upside_per": _up(tp_per), "upside_avg": _up(tp_avg),
            "actual_pbr": actual_pbr, "actual_psr": actual_psr,
            "actual_per": actual_per,
            "pbr_low": pbr_l, "pbr_high": pbr_h,
            "psr_low": psr_l, "psr_high": psr_h,
            "sanity_ratio": sanity_ratio, "sanity_ok": sanity_ok,
        }
        self._mults = out
        return out

    # ─────────────────────────────────────────────────────────
    # 7. DB 저장
    # ─────────────────────────────────────────────────────────
    def save_to_db(self, run_date=None):
        if self.valuation is None:
            return False
        run_date = run_date or datetime.now().strftime("%Y-%m-%d")
        v = self.valuation
        n = self._n
        row = {
            "date": run_date, "ticker": self.ticker_dg,
            "sector": self._sector, "is_excluded": 1 if self._is_excluded else 0,
            "sales_y1": n(v.get("sales_y1")), "sales_y2": n(v.get("sales_y2")),
            "npm_y1":   n(v.get("npm_y1")),   "npm_y2":   n(v.get("npm_y2")),
            "at_y1":    n(v.get("at_y1")),    "at_y2":    n(v.get("at_y2")),
            "fl_y1":    n(v.get("fl_y1")),    "fl_y2":    n(v.get("fl_y2")),
            "roe_y1":   n(v.get("roe_y1")),   "roe_y2":   n(v.get("roe_y2")),
            "ni_y2":    n(v.get("ni_y2")),    "bve_y2":   n(v.get("bve_y2")),
            "eps_y2":   n(v.get("eps_y2")),   "bps_y2":   n(v.get("bps_y2")),
            "sps_y2":   n(v.get("sps_y2")),
            "payout_ratio": n(v.get("payout_ratio")), "g_est": n(v.get("g_est")),
            "beta_raw":    n(v.get("beta_raw")),    "beta_blume":    n(v.get("beta_blume")),
            "beta_sector": n(v.get("beta_sector")), "beta_ensemble": n(v.get("beta_ensemble")),
            "re_low": n(v.get("re_low")), "re_mid": n(v.get("re_mid")),
            "re_high": n(v.get("re_high")),
            "pbr_theory": n(v.get("pbr_theory")), "psr_theory": n(v.get("psr_theory")),
            "per_theory": n(v.get("per_theory")),
            "tp_pbr": n(v.get("tp_pbr")), "tp_psr": n(v.get("tp_psr")),
            "tp_per": n(v.get("tp_per")), "tp_avg": n(v.get("tp_avg")),
            "tp_pbr_low": n(v.get("tp_pbr_low")), "tp_pbr_high": n(v.get("tp_pbr_high")),
            "tp_psr_low": n(v.get("tp_psr_low")), "tp_psr_high": n(v.get("tp_psr_high")),
            "current_price": n(v.get("current_price")),
            "upside_pbr": n(v.get("upside_pbr")), "upside_psr": n(v.get("upside_psr")),
            "upside_per": n(v.get("upside_per")), "upside_avg": n(v.get("upside_avg")),
            "actual_pbr": n(v.get("actual_pbr")), "actual_psr": n(v.get("actual_psr")),
            "actual_per": n(v.get("actual_per")),
            "tax_rate": n(v.get("tax_rate")), "npm_r2": n(v.get("npm_r2")),
            "sanity_ratio": n(v.get("sanity_ratio")),
            "spread_capped": 1 if v.get("g_spread_capped") else 0,
            "forecast_model": self._used_model,
            "forecast_date": self._forecast_date,
        }
        sql = f"""
        INSERT INTO `{TABLE_RESULT}`
        (date,ticker,sector,is_excluded,
         sales_y1,sales_y2,npm_y1,npm_y2,at_y1,at_y2,fl_y1,fl_y2,
         roe_y1,roe_y2,ni_y2,bve_y2,eps_y2,bps_y2,sps_y2,
         payout_ratio,g_est,
         beta_raw,beta_blume,beta_sector,beta_ensemble,
         re_low,re_mid,re_high,
         pbr_theory,psr_theory,per_theory,
         tp_pbr,tp_psr,tp_per,tp_avg,
         tp_pbr_low,tp_pbr_high,tp_psr_low,tp_psr_high,
         current_price,upside_pbr,upside_psr,upside_per,upside_avg,
         actual_pbr,actual_psr,actual_per,tax_rate,npm_r2,
         sanity_ratio,spread_capped,forecast_model,forecast_date)
        VALUES
        (%(date)s,%(ticker)s,%(sector)s,%(is_excluded)s,
         %(sales_y1)s,%(sales_y2)s,%(npm_y1)s,%(npm_y2)s,
         %(at_y1)s,%(at_y2)s,%(fl_y1)s,%(fl_y2)s,
         %(roe_y1)s,%(roe_y2)s,%(ni_y2)s,%(bve_y2)s,
         %(eps_y2)s,%(bps_y2)s,%(sps_y2)s,
         %(payout_ratio)s,%(g_est)s,
         %(beta_raw)s,%(beta_blume)s,%(beta_sector)s,%(beta_ensemble)s,
         %(re_low)s,%(re_mid)s,%(re_high)s,
         %(pbr_theory)s,%(psr_theory)s,%(per_theory)s,
         %(tp_pbr)s,%(tp_psr)s,%(tp_per)s,%(tp_avg)s,
         %(tp_pbr_low)s,%(tp_pbr_high)s,%(tp_psr_low)s,%(tp_psr_high)s,
         %(current_price)s,%(upside_pbr)s,%(upside_psr)s,%(upside_per)s,%(upside_avg)s,
         %(actual_pbr)s,%(actual_psr)s,%(actual_per)s,%(tax_rate)s,%(npm_r2)s,
         %(sanity_ratio)s,%(spread_capped)s,%(forecast_model)s,%(forecast_date)s)
        ON DUPLICATE KEY UPDATE
            tp_pbr=VALUES(tp_pbr), tp_psr=VALUES(tp_psr),
            tp_per=VALUES(tp_per), tp_avg=VALUES(tp_avg),
            upside_avg=VALUES(upside_avg), sector=VALUES(sector),
            roe_y2=VALUES(roe_y2), g_est=VALUES(g_est),
            current_price=VALUES(current_price),
            sanity_ratio=VALUES(sanity_ratio), spread_capped=VALUES(spread_capped)
        """
        conn = get_pymysql_conn(self.db_info)
        try:
            with conn.cursor() as cur:
                cur.execute(sql, row)
            conn.commit()
            return True
        except Exception:
            conn.rollback()
            raise
        finally:
            conn.close()

    # ─────────────────────────────────────────────────────────
    # 8. 전체 실행
    # ─────────────────────────────────────────────────────────
    def run(self):
        self.load_sales()
        self.load_financials()
        self.load_sector()
        beta_re = self.estimate_beta_re()
        dupont  = self.estimate_dupont(beta_re)      # ★ v1.1: beta_re 주입
        mults   = self.compute_multiples(dupont, beta_re)
        self.valuation = {**dupont, **beta_re, **mults}
        if self.verbose:
            v = self.valuation
            tp_s = f"{v['tp_avg']:,.0f}원" if not np.isnan(v["tp_avg"]) else "N/A"
            up_s = f"{v['upside_avg']:+.1f}%" if not np.isnan(v["upside_avg"]) else "N/A"
            log(self.ticker_dg,
                f"ROE_y2={v['roe_y2']:.1%} Re_mid={v['re_mid']:.3%} g={v['g_est']:.2%} | "
                f"PBR={v['pbr_theory']:.2f}x PSR={v['psr_theory']:.2f}x | "
                f"TP_avg={tp_s} Up={up_s}")
        return self


def process_one_ticker_krv(ticker, engine, db_info, rf, e_rm, kospi_series,
                           verbose=False, run_date=None, save_db=True):
    """단일 종목 상대가치 평가 래퍼 (배치/개별 공용)."""
    run_date = run_date or datetime.now().strftime("%Y-%m-%d")
    try:
        m = KoreaRelValModel(ticker=ticker, engine=engine, db_info=db_info,
                             rf=rf, e_rm=e_rm, kospi_series=kospi_series,
                             verbose=verbose)
        m.run()
        saved = m.save_to_db(run_date) if save_db else False
        v = m.valuation
        return {
            "status": "ok", "ticker": m.ticker_dg,
            "sector": m._sector, "excluded": m._is_excluded,
            "tp_pbr": v.get("tp_pbr", np.nan), "tp_psr": v.get("tp_psr", np.nan),
            "tp_per": v.get("tp_per", np.nan), "tp_avg": v.get("tp_avg", np.nan),
            "upside_avg": v.get("upside_avg", np.nan),
            "roe_y2": v.get("roe_y2", np.nan), "re_mid": v.get("re_mid", np.nan),
            "g_est": v.get("g_est", np.nan),
            "saved": saved, "report": m.report, "model": m,
        }
    except Exception as e:
        return {
            "status": "fail", "ticker": to_dg_ticker(ticker),
            "sector": "?", "excluded": False,
            "tp_pbr": np.nan, "tp_psr": np.nan, "tp_per": np.nan,
            "tp_avg": np.nan, "upside_avg": np.nan,
            "roe_y2": np.nan, "re_mid": np.nan, "g_est": np.nan,
            "saved": False, "report": None, "model": None, "msg": str(e)[:150],
        }
    finally:
        gc.collect()


print("[OK] KoreaRelValModel v1.1 정의 완료")
print(f"     g 계산 : calc_sustainable_g() 단일 함수 (estimate_dupont / compute_multiples 공용)")
print(f"     g_cap  : 저성장 GDP({GDP_GROWTH:.1%}) | 중성장 min({G_CAP_MID_FIXED:.0%}, Re x {G_CAP_RE_RATIO}) "
      f"| 고성장 Re x {G_CAP_RE_RATIO}")
print(f"     가드   : Re-g 최소스프레드 {REL_MIN_SPREAD:.2%}  |  Sanity TP/CP {REL_SANITY_TP_RATIO:.0f}배")


## Cell 8 · Excel Export 등록 (6-sheet)

> ⚠️ Cell 7(클래스 셀)을 다시 실행하면 attach 가 풀리므로, 이 셀도 반드시 재실행하세요.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Excel Export · 6-sheet 상대가치 리포트
#  (FCFF 노트북의 export_korea_excel 과 같은 역할 — 상대가치용)
#  - 사용자 입력 없음. 실행하면 KoreaRelValModel 에 메서드가 attach 됨
#  ⚠️ 클래스 셀을 다시 실행하면 attach 가 풀리므로 이 셀도 재실행할 것
# ═══════════════════════════════════════════════════════════════

def export_relval_excel(self, out_dir):
    """상대가치 평가 결과를 6-sheet Excel 로 저장하고 경로를 반환."""
    if self.valuation is None:
        raise RuntimeError(f"[{self.ticker_dg}] run() 미실행 — valuation 없음")

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    v = self.valuation
    fname = f"{self.ticker_dg}_RELVAL.xlsx"
    path = out_dir / fname

    def _fmt(x, unit=""):
        if x is None:
            return None
        try:
            f = float(x)
            return None if (np.isnan(f) or np.isinf(f)) else f
        except (TypeError, ValueError):
            return x

    # ── Sheet 1. Summary ────────────────────────────────────
    sh_summary = pd.DataFrame([
        ("Ticker",              self.ticker_dg),
        ("Sector",              self._sector),
        ("업종 제외 여부",       "제외" if self._is_excluded else "포함"),
        ("매출 예측 모델",       self._used_model),
        ("매출 예측 실행일",     self._forecast_date),
        ("", ""),
        ("현재가 (원)",          _fmt(v.get("current_price"))),
        ("적정주가 PBR (원)",    _fmt(v.get("tp_pbr"))),
        ("적정주가 PSR (원)",    _fmt(v.get("tp_psr"))),
        ("적정주가 PER (원)",    _fmt(v.get("tp_per"))),
        ("적정주가 평균 (원)",   _fmt(v.get("tp_avg"))),
        ("Upside 평균 (%)",      _fmt(v.get("upside_avg"))),
        ("", ""),
        ("이론 PBR (배)",        _fmt(v.get("pbr_theory"))),
        ("이론 PSR (배)",        _fmt(v.get("psr_theory"))),
        ("이론 PER (배)",        _fmt(v.get("per_theory"))),
        ("실제 PBR (배)",        _fmt(v.get("actual_pbr"))),
        ("실제 PSR (배)",        _fmt(v.get("actual_psr"))),
        ("실제 PER (배)",        _fmt(v.get("actual_per"))),
        ("", ""),
        ("Re (mid)",             _fmt(v.get("re_mid"))),
        ("g (지속가능성장률)",   _fmt(v.get("g_est"))),
        ("g_raw (ROE x 유보율)", _fmt(v.get("g_raw"))),
        ("g_cap (적용 상한)",    _fmt(v.get("g_cap"))),
        ("스프레드 가드 적용",   "Y" if v.get("g_spread_capped") else "N"),
        ("Sanity TP/CP (배)",    _fmt(v.get("sanity_ratio"))),
        ("Sanity 통과",          "Y" if v.get("sanity_ok") else "N"),
    ], columns=["항목", "값"])

    # ── Sheet 2. DuPont 2yr ─────────────────────────────────
    sh_dupont = pd.DataFrame({
        "항목": ["매출 (원)", "NOPAT (원)", "NOPAT 마진", "자산회전율(AT)",
                 "재무레버리지(FL)", "ROE", "기말 자기자본 (원)"],
        "Year 1": [_fmt(v.get("sales_y1")), _fmt(v.get("sales_y1", 0) * v.get("npm_y1", 0)),
                   _fmt(v.get("npm_y1")), _fmt(v.get("at_y1")),
                   _fmt(v.get("fl_y1")), _fmt(v.get("roe_y1")), None],
        "Year 2": [_fmt(v.get("sales_y2")), _fmt(v.get("ni_y2")),
                   _fmt(v.get("npm_y2")), _fmt(v.get("at_y2")),
                   _fmt(v.get("fl_y2")), _fmt(v.get("roe_y2")), _fmt(v.get("bve_y2"))],
    })

    # ── Sheet 3. 주당지표 & 멀티플 민감도 ───────────────────
    sh_mult = pd.DataFrame({
        "시나리오": ["Low (β−Δ)", "Mid (β)", "High (β+Δ)"],
        "Re":       [_fmt(v.get("re_low")), _fmt(v.get("re_mid")), _fmt(v.get("re_high"))],
        "이론 PBR": [_fmt(v.get("pbr_low")), _fmt(v.get("pbr_theory")), _fmt(v.get("pbr_high"))],
        "이론 PSR": [_fmt(v.get("psr_low")), _fmt(v.get("psr_theory")), _fmt(v.get("psr_high"))],
        "TP_PBR":   [_fmt(v.get("tp_pbr_low")), _fmt(v.get("tp_pbr")), _fmt(v.get("tp_pbr_high"))],
        "TP_PSR":   [_fmt(v.get("tp_psr_low")), _fmt(v.get("tp_psr")), _fmt(v.get("tp_psr_high"))],
    })

    # ── Sheet 4. 주당지표 ───────────────────────────────────
    sh_ps = pd.DataFrame([
        ("주식수 (자사주차감)", _fmt(v.get("shares"))),
        ("EPS Y2 (원)",         _fmt(v.get("eps_y2"))),
        ("BPS Y2 (원)",         _fmt(v.get("bps_y2"))),
        ("SPS Y2 (원)",         _fmt(v.get("sps_y2"))),
        ("배당성향 (payout)",   _fmt(v.get("payout_ratio"))),
        ("실효세율",            _fmt(v.get("tax_rate"))),
        ("NOPAT마진 회귀 R2",   _fmt(v.get("npm_r2"))),
        ("β_raw",               _fmt(v.get("beta_raw"))),
        ("β_Blume",             _fmt(v.get("beta_blume"))),
        ("β_sector",            _fmt(v.get("beta_sector"))),
        ("β_ensemble",          _fmt(v.get("beta_ensemble"))),
    ], columns=["항목", "값"])

    # ── Sheet 5. 매출 (실적 + 예측) ─────────────────────────
    _act = self._sales_actual.rename("매출_실적").to_frame()
    _fc  = self._sales_forecast.rename("매출_예측").to_frame()
    sh_sales = _act.join(_fc, how="outer")
    sh_sales.index.name = "분기"
    sh_sales = sh_sales.reset_index()

    # ── Sheet 6. 데이터 품질 ────────────────────────────────
    try:
        sh_quality = self.report.to_dataframe()
    except Exception:
        sh_quality = pd.DataFrame({"note": ["품질 리포트 생성 실패"]})

    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        sh_summary.to_excel(xw, sheet_name="1_Summary",     index=False)
        sh_dupont .to_excel(xw, sheet_name="2_DuPont_2yr",  index=False)
        sh_mult   .to_excel(xw, sheet_name="3_Multiple_민감도", index=False)
        sh_ps     .to_excel(xw, sheet_name="4_주당지표_Beta", index=False)
        sh_sales  .to_excel(xw, sheet_name="5_매출_실적예측", index=False)
        sh_quality.to_excel(xw, sheet_name="6_데이터품질",    index=False)

    return str(path)


KoreaRelValModel.export_relval_excel = export_relval_excel
print("[OK] export_relval_excel 등록 완료 (6-sheet: Summary / DuPont / 민감도 / 주당지표 / 매출 / 품질)")


## Cell 9 · 실행 — 종목 리스트 → 평가 & Excel 출력

출력 파일명 끝에 측정일자가 붙습니다 (예: `A005930_RELVAL_20260731.xlsx`).

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  실행 · EXPORT_TICKERS 리스트 → 종목별 상대가치 평가
#  - 사용자 입력 없음 (대상 종목은 Cell 2 의 EXPORT_TICKERS 에서 지정)
#  - ★ 평가는 KoreaRelValModel.run() 단 한 곳에서만 수행 (이중 평가 없음)
#  - EXPORT_EXCEL=True 면 6-sheet Excel, SAVE_TO_DB=True 면 DB 저장
# ═══════════════════════════════════════════════════════════════

if EXPORT_EXCEL:
    assert hasattr(KoreaRelValModel, "export_relval_excel"), \
        "export_relval_excel 없음 — 바로 위 Excel Export 등록 셀을 먼저 실행하세요."

run_date = datetime.now().strftime("%Y-%m-%d")
_date_tag = run_date.replace("-", "")
_summary_rows = []
t0 = time.time()

print(f"\n{'=' * 74}")
print(f"[Individual Relative Valuation v1.1] 대상 {len(EXPORT_TICKERS)}종목  "
      f"run_date={run_date}  SAVE_TO_DB={SAVE_TO_DB}")
print(f"{'=' * 74}")

for i, _tk in enumerate(EXPORT_TICKERS, 1):
    tk = to_dg_ticker(_tk)
    print(f"\n[{i}/{len(EXPORT_TICKERS)}] {tk} " + "─" * 50)
    try:
        _model = KoreaRelValModel(
            ticker=tk, engine=engine, db_info=db_info,
            rf=RF, e_rm=E_RM, kospi_series=KOSPI_PX, verbose=VERBOSE)
        _model.run()
        v = _model.valuation

        # ── Excel 저장 (파일명 끝에 측정일자 부착) ────────────
        _path_str = ""
        if EXPORT_EXCEL:
            _path = Path(_model.export_relval_excel(out_dir=EXPORT_DIR))
            if _date_tag not in _path.stem:
                _new = _path.with_name(f"{_path.stem}_{_date_tag}{_path.suffix}")
                if _new.exists():
                    _new.unlink()
                _path = _path.rename(_new)
            _path_str = str(_path)
            print(f"  ✓ Excel: {_path_str}")

        _saved = _model.save_to_db(run_date) if SAVE_TO_DB else False

        tp_s = f"{v['tp_avg']:,.0f}원" if not np.isnan(v["tp_avg"]) else "N/A"
        cp_s = f"{v['current_price']:,.0f}원" if not np.isnan(v["current_price"]) else "N/A"
        up_s = f"{v['upside_avg']:+.1f}%" if not np.isnan(v["upside_avg"]) else "N/A"
        excl = "  [업종 제외]" if _model._is_excluded else ""

        print(f"    적정주가(평균) : {tp_s}    현재가: {cp_s}    Upside: {up_s}{excl}")
        print(f"    TP_PBR/PSR/PER : "
              f"{v['tp_pbr']:,.0f} / {v['tp_psr']:,.0f} / {v['tp_per']:,.0f} 원")
        print(f"    이론 멀티플    : PBR {v['pbr_theory']:.2f}x  "
              f"PSR {v['psr_theory']:.2f}x  PER {v['per_theory']:.1f}x")
        print(f"    ROE_y2={v['roe_y2']:.1%}  Re_mid={v['re_mid']:.2%}  "
              f"g={v['g_est']:.2%}  (g_raw={v['g_raw']:.2%}, cap={v['g_cap']:.2%})")
        if SAVE_TO_DB:
            print(f"    DB 저장        : {_saved} → {TABLE_RESULT}")

        if SHOW_QUALITY:
            print("\n    ── 데이터 품질 리포트 ──")
            display(_model.report.to_dataframe())

        _summary_rows.append({
            "ticker": tk, "status": "ok", "sector": _model._sector,
            "excluded": _model._is_excluded,
            "tp_pbr": v["tp_pbr"], "tp_psr": v["tp_psr"], "tp_per": v["tp_per"],
            "tp_avg": v["tp_avg"], "current_price": v["current_price"],
            "upside_avg": v["upside_avg"], "roe_y2": v["roe_y2"],
            "re_mid": v["re_mid"], "g_est": v["g_est"], "excel": _path_str,
        })
    except Exception as e:
        print(f"  ✗ FAIL: {e}")
        traceback.print_exc()
        _summary_rows.append({
            "ticker": tk, "status": "fail", "sector": "?", "excluded": False,
            "tp_pbr": np.nan, "tp_psr": np.nan, "tp_per": np.nan,
            "tp_avg": np.nan, "current_price": np.nan, "upside_avg": np.nan,
            "roe_y2": np.nan, "re_mid": np.nan, "g_est": np.nan, "excel": "",
        })
    finally:
        clear_memory()

# ── 최종 요약 ────────────────────────────────────────────────
elapsed = time.time() - t0
print(f"\n{'=' * 74}")
print(f"[완료] {len(_summary_rows)}종목  "
      f"OK={sum(r['status'] == 'ok' for r in _summary_rows)}  "
      f"FAIL={sum(r['status'] == 'fail' for r in _summary_rows)}  "
      f"경과={elapsed:.0f}s")
print(f"{'=' * 74}")
summary_df = pd.DataFrame(_summary_rows).set_index("ticker")
display(summary_df)


## Cell 10 · 종목별 평가 이력 조회 (DB 누적 시)

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  종목별 평가 이력 조회 (DB 에 누적 저장된 경우에만 의미 있음)
# ═══════════════════════════════════════════════════════════════

def get_relval_history(ticker, db_info, table_name=TABLE_RESULT):
    """특정 종목의 날짜별 상대가치 평가 추이."""
    tk = to_dg_ticker(ticker)
    sql = f"""
        SELECT date, sector, roe_y2, re_mid, g_est,
               pbr_theory, psr_theory, per_theory,
               tp_pbr, tp_psr, tp_per, tp_avg,
               current_price, upside_avg,
               actual_pbr, actual_psr, actual_per
        FROM `{table_name}`
        WHERE ticker = %s
        ORDER BY date DESC
    """
    conn = get_pymysql_conn(db_info)
    try:
        with conn.cursor() as cur:
            cur.execute(sql, (tk,))
            return pd.DataFrame(cur.fetchall())
    finally:
        conn.close()


_hist_ticker = EXPORT_TICKERS[0] if EXPORT_TICKERS else "A005930"
print(f"[이력] {_hist_ticker}")
hist = get_relval_history(_hist_ticker, db_info)
display(hist)

if len(hist) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f"{_hist_ticker} — 상대가치 평가 이력", fontsize=12)
    hs = hist.sort_values("date")

    ax = axes[0]
    ax.plot(hs["date"], hs["current_price"], marker="o", label="Current Price",
            color="#95a5a6")
    ax.plot(hs["date"], hs["tp_avg"], marker="s", label="Target Price (avg)",
            color="#e74c3c")
    ax.set_title("Price Path"); ax.set_ylabel("원")
    ax.legend(); ax.grid(alpha=0.3)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

    ax = axes[1]
    ax.plot(hs["date"], hs["upside_avg"], marker="o", color="#2980b9")
    ax.axhline(0, color="black", lw=1)
    ax.axhline(20, color="green", lw=1, ls="--", alpha=0.6)
    ax.axhline(-20, color="red", lw=1, ls="--", alpha=0.6)
    ax.set_title("Upside Path (%)"); ax.grid(alpha=0.3)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

    plt.tight_layout(); plt.show()
